In [1]:
!pip install -q transformers datasets accelerate evaluate jiwer soundfile librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 44.1 MB/s eta 0:00:00


In [2]:
# CELL 2: Imports & Base Model Initialization
import os
import glob
import dataclasses
from typing import Any, Dict, List, Union

import numpy as np
import pandas as pd
import torch
import torchaudio
import evaluate
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = "bangla-speech-processing/BanglaASR"

print(f"Using device: {device}")

# Load processor and base model
processor = WhisperProcessor.from_pretrained(model_path)
model = WhisperForConditionalGeneration.from_pretrained(model_path).to(device)

Using device: cuda


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [3]:
# ==========================================
# CELL 3: Data Parsing & Custom PyTorch Dataset
# ==========================================
dataset_dir = "/kaggle/input/datasets/prosenjitmondol/bangla-regional/shobdotori"
train_dir = os.path.join(dataset_dir, "Train")
annotation_dir = os.path.join(dataset_dir, "Train_annotation")

# 1. Collect all annotation files and map audio paths
csv_files = glob.glob(os.path.join(annotation_dir, "*.csv"))
dfs = []

for csv_path in csv_files:
    district_name = os.path.splitext(os.path.basename(csv_path))[0]
    district_audio_dir = os.path.join(train_dir, district_name)
    sub_df = pd.read_csv(csv_path)
    
    audio_col = sub_df.columns[0]
    text_col = sub_df.columns[1]
    
    sub_df = sub_df[[audio_col, text_col]].copy()
    sub_df.columns = ["audio_id", "sentence"]
    
    def construct_audio_path(val):
        filename = str(val).strip()
        if not filename.lower().endswith(".wav"):
            filename = f"{filename}.wav"
        return os.path.join(district_audio_dir, filename)

    sub_df["audio_path"] = sub_df["audio_id"].apply(construct_audio_path)
    dfs.append(sub_df)

full_df = pd.concat(dfs, ignore_index=True).dropna(subset=["sentence"])
full_df = full_df[full_df["audio_path"].apply(os.path.exists)].reset_index(drop=True)
print(f"Total valid audio samples found: {len(full_df)}")

# 2. Train / Evaluation Split (90/10)
train_df, eval_df = train_test_split(full_df, test_size=0.1, random_state=42)
train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)

# 3. Dynamic PyTorch Dataset (Prevents Jupyter/Kaggle Deadlocks)
class BanglaASRDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        waveform, sample_rate = torchaudio.load(row["audio_path"])
        
        # Convert stereo/multi-channel to mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        # Resample to 16kHz
        if sample_rate != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
            waveform = resampler(waveform)
            
        audio_array = waveform.squeeze().numpy()

        input_features = self.processor.feature_extractor(
            audio_array, sampling_rate=16000
        ).input_features[0]

        labels = self.processor.tokenizer(str(row["sentence"])).input_ids

        return {"input_features": input_features, "labels": labels}

train_dataset = BanglaASRDataset(train_df, processor)
eval_dataset = BanglaASRDataset(eval_df, processor)

Total valid audio samples found: 3350


In [4]:
# ==========================================
# CELL 4: Data Collator & Metric Definition
# ==========================================
@dataclasses.dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [5]:
# ==========================================
# CELL 5: Fine-Tuning Execution
# ==========================================
# FIX: newer `transformers` versions no longer allow controlling generation
# via `model.config`. Setting `suppress_tokens` / `forced_decoder_ids` on
# `model.config` now raises a ValueError as soon as `.generate()` is called
# (which happens automatically at the first eval step, since
# predict_with_generate=True). These must be set on `model.generation_config`
# instead. `use_cache` is a normal (non-generation) config flag, so it stays
# on `model.config`.
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.config.use_cache = False

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/bangla_asr_checkpoints",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=500,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=100,
    eval_steps=100,
    logging_steps=25,
    report_to=["none"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_total_limit=2,
    dataloader_num_workers=2,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

# Resume automatically if a checkpoint already exists in output_dir
# (e.g. if the kernel was interrupted or timed out on a previous run).
def _find_last_checkpoint(output_dir):
    if not os.path.isdir(output_dir):
        return None
    ckpts = glob.glob(os.path.join(output_dir, "checkpoint-*"))
    if not ckpts:
        return None
    return max(ckpts, key=lambda p: int(p.split("-")[-1]))

last_checkpoint = _find_last_checkpoint(training_args.output_dir)
if last_checkpoint:
    print(f"Resuming training from checkpoint: {last_checkpoint}")
trainer.train(resume_from_checkpoint=last_checkpoint)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Wer
100,1.357294,0.320193,53.549469
200,0.281970,0.107336,46.450531
300,0.093042,0.092619,43.376188
400,0.035134,0.088807,40.469536
500,0.009537,0.088466,41.140302


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=500, training_loss=0.7325497440099716, metrics={'train_runtime': 5939.624, 'train_samples_per_second': 2.694, 'train_steps_per_second': 0.084, 'total_flos': 4.58129323008e+18, 'train_loss': 0.7325497440099716, 'epoch': 5.264550264550264})

In [6]:
# ==========================================
# CELL 6: Save Artifacts & Run Validation
# ==========================================
save_path = "/kaggle/working/bangla_asr_best"
trainer.save_model(save_path)
processor.save_pretrained(save_path)
print(f"Saved best model checkpoint to: {save_path}")

# Run evaluation on validation set
eval_results = trainer.evaluate()
print(f"\nFinal Validation Loss: {eval_results.get('eval_loss', 0.0):.4f}")
print(f"Final Validation WER:  {eval_results.get('eval_wer', 0.0):.2f}%")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best model checkpoint to: /kaggle/working/bangla_asr_best


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Final Validation Loss: 0.0888
Final Validation WER:  40.47%


In [7]:
# ==========================================
# CELL 7: Test Inference & Submission Generation
# ==========================================
test_dir = "/kaggle/input/datasets/prosenjitmondol/bangla-regional/shobdotori/Test"
test_audio_files = sorted(glob.glob(os.path.join(test_dir, "**", "*.wav"), recursive=True))

if not test_audio_files:
    test_audio_files = sorted(glob.glob(os.path.join(test_dir, "*.wav")))

print(f"Found {len(test_audio_files)} test audio files. Generating predictions...")

best_model = WhisperForConditionalGeneration.from_pretrained(save_path).to(device)
best_model.eval()

results = []
for file_path in tqdm(test_audio_files):
    file_id = os.path.splitext(os.path.basename(file_path))[0]
    
    waveform, sr = torchaudio.load(file_path)
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
        waveform = resampler(waveform)
        
    audio_array = waveform.squeeze().numpy()
    
    input_features = processor.feature_extractor(
        audio_array, sampling_rate=16000, return_tensors="pt"
    ).input_features.to(device)
    
    with torch.no_grad():
        predicted_ids = best_model.generate(input_features=input_features)
        transcription = processor.decode(predicted_ids[0], skip_special_tokens=True)
        
    results.append({"id": file_id, "transcription": transcription})

submission_df = pd.DataFrame(results)
submission_df.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved predictions to /kaggle/working/submission.csv")
print(submission_df.head())

Found 450 test audio files. Generating predictions...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

  0%|          | 0/450 [00:00<?, ?it/s]

Saved predictions to /kaggle/working/submission.csv
         id                     transcription
0  test_001            তুমি কি খাওয়া আননেছো?
1  test_002         তুমি কি আমাকে কলমটা দেবে?
2  test_003   আজ দুপুরে রাস্তায় পানি জমেছিল।
3  test_004  আজকের সকালে হঠাৎ বৃষ্টি নেমেছিল।
4  test_005          তুমি কি ফরতে একটা পড়বে?


In [ ]:
import os
import glob
import difflib
import json

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import evaluate
from sklearn.metrics import confusion_matrix, roc_curve, auc
from tqdm.auto import tqdm
from transformers import WhisperForConditionalGeneration

plt.rcParams.update({
    "font.size": 12,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

out_dir = "/kaggle/working/paper_results"
os.makedirs(out_dir, exist_ok=True)

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

# ---------- 1. Load the fine-tuned best model ----------
eval_model = WhisperForConditionalGeneration.from_pretrained(save_path).to(device)
eval_model.eval()

# ---------- 2. Run inference on the validation set, capturing confidence ----------
records = []
with torch.no_grad():
    for i in tqdm(range(len(eval_dataset)), desc="Evaluating"):
        item = eval_dataset[i]
        input_features = torch.tensor(item["input_features"]).unsqueeze(0).to(device)

        gen_out = eval_model.generate(
            input_features=input_features,
            max_length=225,
            output_scores=True,
            return_dict_in_generate=True,
        )
        pred_text = processor.decode(gen_out.sequences[0], skip_special_tokens=True)

        # Sequence confidence = mean of the top-token softmax probability at each step
        if gen_out.scores:
            step_probs = [torch.softmax(s, dim=-1).max().item() for s in gen_out.scores]
            confidence = float(np.mean(step_probs))
        else:
            confidence = np.nan

        ref_text = str(eval_df.iloc[i]["sentence"])
        district = os.path.basename(os.path.dirname(eval_df.iloc[i]["audio_path"]))

        try:
            s_wer = wer_metric.compute(predictions=[pred_text], references=[ref_text])
            s_cer = cer_metric.compute(predictions=[pred_text], references=[ref_text])
        except Exception:
            s_wer, s_cer = np.nan, np.nan

        records.append({
            "audio_path": eval_df.iloc[i]["audio_path"],
            "district": district,
            "reference": ref_text,
            "prediction": pred_text,
            "wer": s_wer,
            "cer": s_cer,
            "confidence": confidence,
            "correct": int(s_wer == 0.0) if pd.notna(s_wer) else np.nan,
        })

results_df = pd.DataFrame(records)
results_df.to_csv(f"{out_dir}/per_sample_results.csv", index=False)

# ---------- 3. Corpus-level metrics ----------
valid = results_df.dropna(subset=["wer", "cer"])
overall_wer = 100 * wer_metric.compute(predictions=valid["prediction"].tolist(), references=valid["reference"].tolist())
overall_cer = 100 * cer_metric.compute(predictions=valid["prediction"].tolist(), references=valid["reference"].tolist())
sentence_acc = 100 * valid["correct"].mean()

metrics_summary = {
    "word_error_rate_pct": round(overall_wer, 2),
    "word_accuracy_pct": round(100 - overall_wer, 2),
    "character_error_rate_pct": round(overall_cer, 2),
    "character_accuracy_pct": round(100 - overall_cer, 2),
    "sentence_exact_match_accuracy_pct": round(sentence_acc, 2),
    "mean_per_sample_wer_pct": round(valid["wer"].mean() * 100, 2),
    "median_per_sample_wer_pct": round(valid["wer"].median() * 100, 2),
    "num_eval_samples": len(valid),
}

with open(f"{out_dir}/metrics_summary.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)

print("=" * 55)
print("CORPUS-LEVEL METRICS (for paper's Results table)")
print("=" * 55)
for k, v in metrics_summary.items():
    print(f"{k:38s}: {v}")

# ---------- 4. Figures ----------
# 4a. Train / validation loss curve
log_history = pd.DataFrame(trainer.state.log_history)
train_log = log_history[log_history.get("loss").notna()] if "loss" in log_history else pd.DataFrame()
eval_log = log_history[log_history.get("eval_loss").notna()] if "eval_loss" in log_history else pd.DataFrame()

fig, ax = plt.subplots(figsize=(6, 4.5))
if not train_log.empty:
    ax.plot(train_log["step"], train_log["loss"], label="Train loss", color="tab:blue")
if not eval_log.empty:
    ax.plot(eval_log["step"], eval_log["eval_loss"], label="Val loss", color="tab:orange", marker="o")
ax.set_xlabel("Step"); ax.set_ylabel("Loss"); ax.set_title("Training / Validation Loss")
ax.legend()
fig.tight_layout(); fig.savefig(f"{out_dir}/loss_curve.png"); plt.show()

# 4b. Validation WER over training
fig, ax = plt.subplots(figsize=(6, 4.5))
if not eval_log.empty and "eval_wer" in eval_log:
    ax.plot(eval_log["step"], eval_log["eval_wer"], marker="o", color="tab:green")
ax.set_xlabel("Step"); ax.set_ylabel("WER (%)"); ax.set_title("Validation WER over Training")
fig.tight_layout(); fig.savefig(f"{out_dir}/wer_curve.png"); plt.show()

# 4c. Distribution of per-sample WER
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.hist(valid["wer"] * 100, bins=30, color="tab:purple", edgecolor="black")
ax.set_xlabel("Per-sample WER (%)"); ax.set_ylabel("Count"); ax.set_title("Distribution of Per-Sample WER")
fig.tight_layout(); fig.savefig(f"{out_dir}/wer_distribution.png"); plt.show()

# 4d. ROC curve: confidence -> exact-match correctness
fig, ax = plt.subplots(figsize=(5.5, 5))
if valid["confidence"].notna().sum() > 1 and valid["correct"].nunique() > 1:
    fpr, tpr, _ = roc_curve(valid["correct"], valid["confidence"])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color="tab:red", label=f"AUC = {roc_auc:.3f}")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC: Confidence \u2192 Exact-Match Correctness")
    ax.legend()
    metrics_summary["roc_auc"] = round(roc_auc, 3)
else:
    ax.text(0.5, 0.5, "Not enough variation\nto compute ROC", ha="center", va="center")
    ax.set_title("ROC Curve")
fig.tight_layout(); fig.savefig(f"{out_dir}/roc_curve.png"); plt.show()

# 4e. Character-substitution confusion matrix (top-15 most frequent chars)
def char_substitutions(ref, hyp):
    sm = difflib.SequenceMatcher(None, ref, hyp)
    subs = []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "replace":
            for rc, hc in zip(ref[i1:i2], hyp[j1:j2]):
                subs.append((rc, hc))
    return subs

all_subs = []
for _, row in valid.iterrows():
    all_subs.extend(char_substitutions(row["reference"], row["prediction"]))
sub_df = pd.DataFrame(all_subs, columns=["true_char", "pred_char"])
sub_df.to_csv(f"{out_dir}/confusion_pairs.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 6))
if not sub_df.empty:
    top_chars = pd.concat([sub_df["true_char"], sub_df["pred_char"]]).value_counts().head(15).index.tolist()
    filt = sub_df[sub_df["true_char"].isin(top_chars) & sub_df["pred_char"].isin(top_chars)]
    if not filt.empty:
        cm = confusion_matrix(filt["true_char"], filt["pred_char"], labels=top_chars)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Reds", ax=ax,
                    xticklabels=top_chars, yticklabels=top_chars, cbar=False)
        ax.set_xlabel("Predicted Character"); ax.set_ylabel("True Character")
        ax.set_title("Character Substitution Confusion Matrix (Top 15)")
    else:
        ax.text(0.5, 0.5, "No overlapping\nsubstitutions", ha="center", va="center")
else:
    ax.text(0.5, 0.5, "No substitution errors found", ha="center", va="center")
    ax.set_title("Character Substitution Confusion Matrix")
fig.tight_layout(); fig.savefig(f"{out_dir}/confusion_matrix.png"); plt.show()

# 4f. WER by district (regional breakdown)
fig, ax = plt.subplots(figsize=(6, 5))
dist_wer = valid.groupby("district")["wer"].mean().sort_values() * 100
dist_wer.plot(kind="barh", ax=ax, color="tab:cyan")
ax.set_xlabel("Mean WER (%)"); ax.set_title("WER by District")
fig.tight_layout(); fig.savefig(f"{out_dir}/wer_by_district.png"); plt.show()

with open(f"{out_dir}/metrics_summary.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)

print(f"\nAll paper artifacts saved to: {out_dir}")
print("  per_sample_results.csv, confusion_pairs.csv, metrics_summary.json")
print("  loss_curve.png, wer_curve.png, wer_distribution.png,")
print("  roc_curve.png, confusion_matrix.png, wer_by_district.png")